**Setup**

Import the required libraries and load the chatbot conversation dataset.

* Dataset: bitext/customer-support-llm-chatbot-training-dataset (Modified)
* Original: https://huggingface.co/datasets/bitext/customer-support-llm-chatbot-training-dataset

MODIFICATIONS (Educational):
- Added synthetic PII columns and injected PII into text
- Introduced data quality issues
- Purpose: Simulate realistic data preprocessing challenges


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load the chatbot conversation dataset
df = pd.read_csv('./data/customer_support_clean.csv')

**Example**

You'll work with production conversation logs from your AI chatbot. Each row represents one conversation with metrics like response time, cost, and quality indicators. Explore a sample conversation:

In [3]:
sample = df.iloc[100]
print("=== Sample Conversation ===")
#print(f"Conversation ID: {sample['conversation_id']}")
print(f"Model used: {sample['model_used']}")
print(f"Latency: {sample['latency_seconds']:.2f}s")
print(f"Tokens used: {sample['tokens_used']}")
print(f"Cost: ${sample['cost']:.4f}")
print(f"Resolved: {sample['resolved']}")
print(f"CSAT: {sample['csat_score']}/5")
print(f"Hallucinated: {sample['hallucinated']}")
print(f"Columns: {list(sample.index)}")

=== Sample Conversation ===
Model used: gpt-4
Latency: 2.78s
Tokens used: 194
Cost: $0.0058
Resolved: True
CSAT: 3.1/5
Hallucinated: False
Columns: ['customer_id_hashed', 'customer_type', 'zipcode', 'week_number', 'flags', 'instruction', 'response', 'category', 'intent', 'model_tier', 'model_used', 'tokens_used', 'cost', 'latency_seconds', 'conversation_turns', 'resolved', 'escalated_to_human', 'csat_score', 'hallucinated', 'error_occurred']


#### Detect Performance Disparities by Customer Type

Your chatbot's overall metrics look good, but does it work equally well for all customer types? New customers might have different needs than returning customers, and the AI might have learned to serve one group better than the other.

1. Calculate the resolution rate, average CSAT score, and escalation rate for each `customer_type` and save in `metrics_by_type`
2. Compare each group to the overall average by computing the performance gap of each metric; for example, add the column `resolution_gap` to `metrics_by_type` that calculates the percent difference of the customer type resolution rate from the average resolution rate overall
3. Print the results and verify if any metrics and groups fall outside the 10% range


**The 10% Rule:** If any group performs more than 10% worse than the average, it indicates a fairness issue that needs investigation.

In [4]:
# Calculate overall averages (baseline)
overall_resolution = df['resolved'].mean()
overall_csat = df['csat_score'].mean()
overall_escalation = df['escalated_to_human'].mean()

# Calculate metrics by customer type
metrics_by_type = df.groupby('customer_type').agg(
    resolution_rate=pd.NamedAgg(column='resolved', aggfunc='mean'),
    avg_csat=pd.NamedAgg(column='csat_score', aggfunc='mean'),
    escalation_rate=pd.NamedAgg(column='escalated_to_human', aggfunc='mean')
).reset_index()

# Calculate performance gaps (difference from average)
metrics_by_type['resolution_gap'] = (
    (metrics_by_type['resolution_rate'] - overall_resolution)
    / overall_resolution * 100
)

metrics_by_type['csat_gap'] = (
    (metrics_by_type['avg_csat'] - overall_csat)
    / overall_csat * 100
)

metrics_by_type['escalation_gap'] = (
    (metrics_by_type['escalation_rate'] - overall_escalation)
    / overall_escalation * 100
)

print("=== Fairness Analysis by Customer Type ===")
print(f"\nOverall Baseline:")
print(f"  Resolution rate: {overall_resolution:.1%}")
print(f"  CSAT score: {overall_csat:.2f}/5")
print(f"  Escalation rate: {overall_escalation:.1%}")

print(f"\n{'='*70}")
print(f"Performance by Customer Type:")
print(f"{'='*70}")

print(metrics_by_type)

=== Fairness Analysis by Customer Type ===

Overall Baseline:
  Resolution rate: 73.1%
  CSAT score: 4.05/5
  Escalation rate: 8.2%

Performance by Customer Type:
  customer_type  resolution_rate  avg_csat  escalation_rate  resolution_gap  \
0    enterprise         0.734559  4.048945         0.081500        0.437891   
1           new         0.731700  4.051754         0.081367        0.047020   
2       premium         0.733284  4.045799         0.084467        0.263615   
3     returning         0.726027  4.046124         0.079277       -0.728600   

   csat_gap  escalation_gap  
0  0.020282       -0.178663  
1  0.089670       -0.341855  
2 -0.057449        3.455308  
3 -0.049427       -2.901716  


#### Test Statistical Significance and Analyze by Category

None of your performance differences fell outside the 10% guideline, which is great! But it's still a good idea to verify if the performance differences are statistically significant or not.

1. Run a chi-square test on resolution rates by customer type to test statistical significance
   - Use `stats.chi2_contingency()` with a contingency table of resolved/unresolved counts
   - If p-value < 0.05, the difference is statistically significant (likely real, not random)
2. Calculate the resolution rate for each issue category, `category`
3. Identify any categories with resolution rates 10% _worse_ than the average and print the type(s)

**Understanding chi-square:** This test answers: "Is the difference in resolution rates large enough that it's unlikely to be random chance?"

In [7]:
# Statistical significance test for customer type
print("=== Statistical Significance Test: Customer Type ===")
print("\nTesting if resolution rate differences are statistically significant...\n")

# Create contingency table: rows=customer_type, columns=resolved/unresolved
contingency_type = pd.crosstab(df['customer_type'], df['resolved'])
print("Contingency Table (Customer Type vs Resolution):")
print(contingency_type)

# Run chi-square test
chi2_type, p_value_type, dof_type, expected_type = stats.chi2_contingency(contingency_type)

print(f"\nChi-square statistic: {chi2_type:.4f}")
print(f"P-value: {p_value_type:.4f}")
print(f"Degrees of freedom: {dof_type}")

# Analyze by issue category
print(f"\n\n{'='*70}")
print(f"=== Fairness Analysis by Issue Category ===")
print(f"{'='*70}")

overall_resolution = df['resolved'].mean()

metrics_by_category = df.groupby('error_occurred').agg(
    resolution_rate=pd.NamedAgg(column='resolved', aggfunc='mean'),
    count=pd.NamedAgg(column='resolved', aggfunc='count')
).reset_index()

# Calculate gaps
metrics_by_category['resolution_gap'] = (
    (metrics_by_category['resolution_rate'] - overall_resolution)
    / overall_resolution * 100
)

# Check for critically underperforming types
critical_category = metrics_by_category[metrics_by_category['resolution_gap'] < -10]

if len(critical_category) > 0:
    print(f"\nAffected Categories:")
    for category, row in critical_category.iterrows():
        print(f"   • {category}: {row['resolution_rate']:.1%} resolution ({row['count']} conversations)")



=== Statistical Significance Test: Customer Type ===

Testing if resolution rate differences are statistically significant...

Contingency Table (Customer Type vs Resolution):
resolved       False  True 
customer_type              
enterprise      1762   4876
new             1774   4838
premium         1803   4957
returning       1880   4982

Chi-square statistic: 1.4701
P-value: 0.6892
Degrees of freedom: 3


=== Fairness Analysis by Issue Category ===


#### Geographic Bias and Mitigation Recommendations

ZIP codes can represent race, income, and other protected characteristics. If our chatbot performs worse in certain ZIP codes, we might be accidentally discriminating. We need to identify geographic differences and propose solutions.

1. Calculate the resolution rate by ZIP codes
2. Identify any ZIP codes performing 10% _worse_ than the average and print the ZIP code(s) with their resolution rates and counts

In [8]:
## YOUR SOLUTION HERE ##
print("\nAnalyzing performance by ZIP code\n")

# Calculate metrics by ZIP code
metrics_by_zip = df.groupby('zipcode').agg(
    resolution_rate=pd.NamedAgg(column='resolved', aggfunc='mean'),
    count=pd.NamedAgg(column='resolved', aggfunc='count')
).reset_index()

# Calculate gaps
metrics_by_zip['resolution_gap'] = (
    (metrics_by_zip['resolution_rate'] - overall_resolution)
    / overall_resolution * 100
)

# Check for critically underperforming ZIP codes
critical_zips = metrics_by_zip[metrics_by_zip['resolution_gap'] < -10]

if len(critical_zips) > 0:
    print(f"\nAffected ZIP codes:")
    for zipcode, row in critical_zips.iterrows():
        print(f"   • {zipcode}: {row['resolution_rate']:.1%} resolution ({row['count']} conversations)")



Analyzing performance by ZIP code


Affected ZIP codes:
   • 17: 0.0% resolution (1.0 conversations)
